In [ ]:
from pathlib import Path
import os
import shutil
from typing import Iterable, Tuple


### Local destination on either computer

Run this setup before the examples. It selects an existing OneDrive `Lab/Data` folder from Windows OneDrive settings, your user folder, or `D:/Alejandro/Data`. If none is available, it reports the checked paths without creating folders.

Set each example's `dst_root` below this base: `DATA_ROOT / "2p"`, or `DATA_ROOT / "others" / "Clara"` for the Clara data. Copy examples start with `dry_run=True`; use `False` only after reviewing the source and destination.


In [ ]:
# Candidate OneDrive locations for both computers, in preference order.
ONEDRIVE_ROOTS = [
    Path(os.environ[key])
    for key in ("OneDriveCommercial", "OneDrive")
    if os.environ.get(key)
] + [
    Path.home() / "OneDrive - Université de Lausanne",
    Path(r"D:\Alejandro\Data") / "OneDrive - Université de Lausanne",
]
DATA_ROOT_CANDIDATES = [root / "Lab" / "Data" for root in ONEDRIVE_ROOTS]
DATA_ROOT = next((path for path in DATA_ROOT_CANDIDATES if path.is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError(
        "No local OneDrive Lab/Data folder is available. Check OneDrive or edit "
        "ONEDRIVE_ROOTS. Tried: " + "; ".join(map(str, DATA_ROOT_CANDIDATES))
    )
print(f"Local data root: {DATA_ROOT}")


In [ ]:
# Prints the relative paths under root up to max_depth.
#Use this to sanity-check that renames/moves produced the expected structure.
root = DATA_ROOT / "2p" / "L453_f10_Exp_1_flickering"
max_depth = 3
# change this to show deeper levels

for p in root.rglob("*"):
    depth = len(p.relative_to(root).parts)
    if depth <= max_depth:
        print(p.relative_to(root))


In [ ]:
# Helpers to copy only 03_analysis from each experiment
# \
# Defines:
# find_experiments_with_analysis(src_root): yields experiments that contain a 03_analysis folder.
# copy_analysis_only(...): copies each experiment’s 03_analysis into the destination, with:
# dry_run=True → prints what it would do (no copy)
# overwrite=True → removes an existing destination 03_analysis before copying
# Returns a tuple: (n_experiments_found, n_operations_{planned|done}).

def find_experiments_with_analysis(src_root: Path) -> Iterable[Path]:
    """
    Yield experiment directories under src_root that contain a '03_analysis' subfolder.
    """
    for p in src_root.iterdir():
        if p.is_dir() and (p / "03_analysis").exists():
            yield p


def copy_analysis_only(
        src_root: Path,
        dst_root: Path,
        exp_glob: str = "*",  # pattern to match experiment folder names, e.g. "L4*_Exp_*"
        dry_run: bool = True,  # True = preview; False = actually copy
        overwrite: bool = False,  # True = allow overwriting an existing destination
) -> Tuple[int, int]:
    """
    Copy only the '03_analysis' folder for each experiment folder from src_root to dst_root.

    Layout copied:
      dst_root/<experiment_name>/03_analysis/<...contents...>

    Returns (n_experiments, n_operations_planned).
    """
    src_root = Path(src_root)
    dst_root = Path(dst_root)
    assert src_root.exists(), f"Source root does not exist: {src_root}"

    exps = [p for p in find_experiments_with_analysis(src_root) if p.match(exp_glob)]
    if not exps:
        print("No experiment folders with '03_analysis' found matching pattern.")
        return 0, 0

    ops = 0
    for exp_dir in exps:
        src_analysis = exp_dir / "03_analysis"
        dst_exp = dst_root / exp_dir.name
        dst_analysis = dst_exp / "03_analysis"

        # Plan: create <dst_exp>, then copytree 03_analysis → dst_analysis
        print(f"\n[EXP] {exp_dir.name}")
        print(f"  SRC: {src_analysis}")
        print(f"  DST: {dst_analysis}")

        if dry_run:
            if dst_analysis.exists():
                action = "would OVERWRITE" if overwrite else "would SKIP (exists)"
            else:
                action = "would COPY"
            print(f"  Action (dry-run): {action}")
            ops += 1
            continue

        # Actual copy
        dst_exp.mkdir(parents=True, exist_ok=True)
        if dst_analysis.exists():
            if overwrite:
                print("  Removing existing destination (overwrite=True)...")
                shutil.rmtree(dst_analysis)
            else:
                print("  Destination exists and overwrite=False; skipping.")
                continue

        print("  Copying...")
        shutil.copytree(src_analysis, dst_analysis)  # copies full subtree
        ops += 1

    print(f"\nDone. Experiments found: {len(exps)} | Operations {'planned' if dry_run else 'done'}: {ops}")
    return len(exps), ops


In [ ]:
#copy_analysis_only from network to local

# copy_analysis_only from network to local

# Example: preview first, then optionally execute the 03_analysis copy.
#  1) Leave dry_run=True to print the planned actions without copying files.
#  2) After checking the paths, use the commented execute block with dry_run=False.
# overwrite=False keeps any existing destination; set True only if you want to replace it.
# Set your source and destination roots (Windows UNC / mapped drive paths).

src_root = Path(r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\07_Data\Alejandro")
dst_root = DATA_ROOT / "2p"  # For Clara: DATA_ROOT / "others" / "Clara"

# 1) Preview (no files copied)
copy_analysis_only(
    src_root=src_root,
    dst_root=dst_root,
    exp_glob="L800_f08",  # or e.g. "L4*_Exp_*" to limit which experiments
    dry_run=True,  # preview only
    overwrite=False,
)
# # 2) Execute copy only after the preview output looks correct
# copy_analysis_only(
#     src_root=src_root,
#     dst_root=dst_root,
#     exp_glob="*",  # same pattern you previewed
#     dry_run=False,  # actually copy
#     overwrite=False,  # set True if you want to replace existing destinations
# )


In [ ]:
# Scratch-only network path check.
# This is only for ad hoc debugging and can be deleted later.
# Uncomment the lines below only if you need to verify that the source path is reachable.
# src_root = Path(r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\07_Data\Alejandro")
# print("Exists:", src_root.exists())


In [ ]:
#copy_metadata from network to local

# Function to copy only 01_raw/2p/metadata/**

# Defines copy_metadata_only(...):

# Finds experiments in src_root matching exp_glob.
# If dry_run=True, prints the action it would take.
# If executing, recreates the internal folder structure and copies files into …/01_raw/2p/metadata.
# Respects overwrite: if False, existing files are left as-is.

def copy_metadata_only(
    src_root: Path,
    dst_root: Path,
    exp_glob: str = "*",
    dry_run: bool = True,
    overwrite: bool = False,
):
    """
    Copy only 01_raw/2p/metadata/** for experiments that contain it.
    """
    src_root = Path(src_root); dst_root = Path(dst_root)
    exps = [p for p in src_root.iterdir() if p.is_dir() and p.match(exp_glob)]
    ops = 0
    for exp_dir in exps:
        src_meta = exp_dir / "01_raw" / "2p" / "metadata"
        if not src_meta.exists():
            continue
        dst_meta = dst_root / exp_dir.name / "01_raw" / "2p" / "metadata"
        print(f"\n[EXP] {exp_dir.name}")
        print(f"  SRC: {src_meta}")
        print(f"  DST: {dst_meta}")
        if dry_run:
            if dst_meta.exists():
                action = "would MERGE CONTENTS" if not overwrite else "would OVERWRITE CONTENTS"
            else:
                action = "would COPY CONTENTS"
            print(f"  Action (dry-run): {action}")
            ops += 1
        else:
            dst_meta.mkdir(parents=True, exist_ok=True)
            for root, _, files in os.walk(src_meta):
                rel = Path(root).relative_to(src_meta)
                here = dst_meta / rel
                here.mkdir(parents=True, exist_ok=True)
                for f in files:
                    s = Path(root) / f
                    d = here / f
                    if d.exists() and not overwrite:
                        continue
                    shutil.copy2(s, d)
            ops += 1
    print(f"\nDone. Operations {'planned' if dry_run else 'done'}: {ops}")


In [ ]:
# Example: preview first, then optionally execute the metadata copy

# Calls copy_metadata_only(...) with your src_root/dst_root.
#  1) Leave dry_run=True to print the planned actions without copying files.
#  2) Change to dry_run=False only when you are ready to copy.
# overwrite=False prevents replacing existing files; set to True only if you need to refresh them.

src_root = Path(r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\07_Data\Alejandro")
dst_root = DATA_ROOT / "2p"  # For Clara: DATA_ROOT / "others" / "Clara"

# Preview only (prints what it will do; copies nothing)
copy_metadata_only(
    src_root=src_root,
    dst_root=dst_root,
    exp_glob="L800*",   # pattern for experiment folder names (use "*" for all)
    dry_run=True,     # preview only
    overwrite=False  # don't replace files if they already exist in dst
)


### Copy anatomy folders only

Copy `<experiment>/01_raw/2p/anatomy` with all its contents, preserving that layout locally. Experiments without anatomy are reported and skipped. Existing anatomy folders are skipped with `overwrite=False`; `overwrite=True` merges the source contents and replaces matching files, retaining other destination files.

Run the imports and local-data setup above first. Preview with `dry_run=True`, review the paths, then use `dry_run=False` to copy.


In [ ]:
def copy_anatomy_only(
    src_root: Path,
    dst_root: Path,
    exp_glob: str = "*",
    dry_run: bool = True,
    overwrite: bool = False,
) -> Tuple[int, int]:
    """Copy experiment/01_raw/2p/anatomy; return (matched, planned_or_copied).

    Missing anatomy and existing destinations (unless overwrite=True) are skipped.
    A preview creates no files or directories. Overwrite merges without deletion.
    """
    src_root, dst_root = Path(src_root), Path(dst_root)
    if not src_root.is_dir():
        raise FileNotFoundError(f"Source root is not an accessible directory: {src_root}")
    if dst_root.resolve().is_relative_to(src_root.resolve()):
        raise ValueError("Destination root must be outside the source experiment tree.")
    experiments = sorted(p for p in src_root.iterdir() if p.is_dir() and p.match(exp_glob))
    relative_anatomy = Path("01_raw") / "2p" / "anatomy"
    operations = 0
    missing = 0
    existing = 0
    for experiment in experiments:
        source = experiment / relative_anatomy
        destination = dst_root / experiment.name / relative_anatomy
        if not source.is_dir():
            print(f"[SKIP] {experiment.name}: no {relative_anatomy} folder")
            missing += 1
            continue
        if destination.exists() and not overwrite:
            print(f"[SKIP] {experiment.name}: anatomy destination exists (overwrite=False)")
            existing += 1
            continue
        action = "MERGE/REPLACE matching files" if destination.exists() else "COPY"
        print(f"[EXP] {experiment.name}\n  SRC: {source}\n  DST: {destination}")
        print(f"  {'Would' if dry_run else 'Will'} {action}")
        if not dry_run:
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copytree(source, destination, dirs_exist_ok=overwrite)
        operations += 1
    print(
        f"Matched: {len(experiments)} | {'Planned' if dry_run else 'Copied'}: {operations} "
        f"| Missing anatomy: {missing} | Existing skipped: {existing}"
    )
    return len(experiments), operations


### Preview Clara anatomy copy

The source is Clara's network `Microscopy` folder. The destination uses the local OneDrive root detected for this computer. Start with one experiment, such as `exp_glob="LC001_f01"`, if you want a smaller preview; `"LC*"` selects all matching experiments.


In [ ]:
ANATOMY_SRC_ROOT = Path(
    r"\\nasdcsr.unil.ch\RECHERCHE\FAC\FBM\CIG\jlarsch\default\D2c\07_Data\Clara\Microscopy"
)
ANATOMY_DST_ROOT = DATA_ROOT / "others" / "Clara"

copy_anatomy_only(
    src_root=ANATOMY_SRC_ROOT,
    dst_root=ANATOMY_DST_ROOT,
    exp_glob="LC001_f05",
    dry_run=False,  # Change to False after reviewing the preview.
    overwrite=True,
)
